# Snowflake Table Types (A-Z)

A comprehensive reference of all table types in Snowflake with creation scripts and pro tips.

| # | Table Type | Time Travel | Fail-safe | Use Case |
|---|-----------|-------------|-----------|----------|
| 1 | Directory Table | N/A | N/A | Auto-catalog files on a stage |
| 2 | Dynamic Table | Yes (1 day) | Yes (7 days) | Declarative data pipelines (replaces streams+tasks) |
| 3 | Event Table | Yes (1 day) | Yes (7 days) | Capture telemetry/log events from UDFs, SPs, Streamlit |
| 4 | External Table | No | No | Query data in external stages (S3, GCS, Azure) without loading |
| 5 | Hybrid Table | Yes (1 day) | Yes (7 days) | Low-latency OLTP workloads with primary key enforcement |
| 6 | Iceberg Table | Yes (1 day) | Yes (7 days) | Open table format, interoperable with Spark/Flink/Trino |
| 7 | Permanent Table | Yes (0-90 days) | Yes (7 days) | Default — production data, persistent storage |
| 8 | Temporary Table | Yes (0-1 day) | No | Session-scoped, auto-dropped on session end |
| 9 | Transient Table | Yes (0-1 day) | No | Staging/ETL — no fail-safe cost |

In [ ]:
%%sql -r setup
CREATE DATABASE IF NOT EXISTS PRACTICE;
CREATE SCHEMA IF NOT EXISTS PRACTICE.TABLE_TYPES;

---
## 1. Directory Table

In [ ]:
%%sql -r directory_tbl
CREATE OR REPLACE STAGE PRACTICE.TABLE_TYPES.my_directory_stage
    DIRECTORY = (ENABLE = TRUE);

-- After uploading files, refresh to update the directory metadata
-- ALTER STAGE PRACTICE.TABLE_TYPES.my_directory_stage REFRESH;

-- Query the directory table
-- SELECT * FROM DIRECTORY(@PRACTICE.TABLE_TYPES.my_directory_stage);

---
## 2. Dynamic Table

In [ ]:
%%sql -r dynamic_tbl
CREATE OR REPLACE TABLE PRACTICE.TABLE_TYPES.raw_orders (
    order_id    NUMBER,
    customer_id NUMBER,
    amount      DECIMAL(10,2),
    order_date  DATE
);

INSERT INTO PRACTICE.TABLE_TYPES.raw_orders VALUES
(1, 101, 250.00, '2026-04-01'),
(2, 102, 175.50, '2026-04-02'),
(3, 101, 320.00, '2026-04-03');

CREATE OR REPLACE DYNAMIC TABLE PRACTICE.TABLE_TYPES.dt_customer_totals
    TARGET_LAG   = '1 hour'
    WAREHOUSE    = COMPUTE_WH
    REFRESH_MODE = AUTO
    INITIALIZE   = ON_CREATE
    DATA_RETENTION_TIME_IN_DAYS = 1
    COMMENT      = 'Aggregated customer order totals — auto-refreshed hourly'
AS
    SELECT
        customer_id,
        COUNT(*)        AS total_orders,
        SUM(amount)     AS total_amount,
        MAX(order_date) AS last_order_date
    FROM PRACTICE.TABLE_TYPES.raw_orders
    GROUP BY customer_id;

---
## 3. Event Table

In [ ]:
%%sql -r event_tbl
CREATE OR REPLACE EVENT TABLE PRACTICE.TABLE_TYPES.my_events
    DATA_RETENTION_TIME_IN_DAYS = 30
    CHANGE_TRACKING = TRUE
    COMMENT = 'Account-level telemetry: logs, traces, metrics from UDFs/SPs';

In [ ]:
%%sql -r set_event_tbl
ALTER ACCOUNT SET EVENT_TABLE = 'PRACTICE.TABLE_TYPES.MY_EVENTS';
ALTER ACCOUNT SET LOG_LEVEL = 'INFO';

In [ ]:
%%sql -r create_logging_sp
CREATE OR REPLACE PROCEDURE PRACTICE.TABLE_TYPES.SP_DEMO_LOGGING()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import logging

logger = logging.getLogger('dq_demo_logger')

def run(session):
    logger.info('SP started — beginning data quality check')

    try:
        row_count = session.sql('SELECT COUNT(*) AS CNT FROM PRACTICE.TABLE_TYPES.PERM_EMPLOYEES').collect()[0]['CNT']
        logger.info(f'Table has {row_count} rows',
                    extra={'table': 'PERM_EMPLOYEES', 'row_count': row_count})

        if row_count == 0:
            logger.warning('Table is empty — possible data issue!')
            return 'WARNING: No data found'

        logger.info('Checking for null values in SALARY column')
        null_count = session.sql("SELECT COUNT(*) AS CNT FROM PRACTICE.TABLE_TYPES.PERM_EMPLOYEES WHERE SALARY IS NULL").collect()[0]['CNT']

        if null_count > 0:
            logger.error(f'Found {null_count} null salary values!',
                         extra={'check': 'NULL_CHECK', 'column': 'SALARY', 'nulls': null_count})
        else:
            logger.info('No null salaries found — PASS',
                        extra={'check': 'NULL_CHECK', 'column': 'SALARY', 'status': 'PASS'})

        logger.info('SP completed successfully')
        return f'SUCCESS: {row_count} rows checked, {null_count} null salaries'

    except Exception as e:
        logger.error(f'SP failed with error: {str(e)}')
        return f'ERROR: {str(e)}'
$$;

ALTER PROCEDURE PRACTICE.TABLE_TYPES.SP_DEMO_LOGGING() SET LOG_LEVEL = 'INFO';

In [ ]:
%%sql -r call_logging_sp
CALL PRACTICE.TABLE_TYPES.SP_DEMO_LOGGING();

In [ ]:
%%sql -r query_event_logs
SELECT
    TIMESTAMP                              AS EVENT_TIME,
    RECORD['severity_text']::VARCHAR       AS LOG_LEVEL,
    VALUE::VARCHAR                         AS MESSAGE,
    SCOPE['name']::VARCHAR                 AS LOGGER_NAME,
    RECORD_ATTRIBUTES                      AS CUSTOM_ATTRIBUTES,
    RESOURCE_ATTRIBUTES['snow.executable.name']::VARCHAR AS OBJECT_NAME
FROM PRACTICE.TABLE_TYPES.MY_EVENTS
WHERE RECORD_TYPE = 'LOG'
  AND SCOPE['name'] = 'dq_demo_logger'
ORDER BY TIMESTAMP DESC
LIMIT 20;

---
## 4. External Table

In [ ]:
%%sql -r external_tbl
-- External table requires an external stage pointing to cloud storage.
-- Example using S3:

-- CREATE OR REPLACE STAGE PRACTICE.TABLE_TYPES.my_s3_stage
--     URL = 's3://my-bucket/data/'
--     CREDENTIALS = (AWS_KEY_ID = '...' AWS_SECRET_KEY = '...');

-- CREATE OR REPLACE EXTERNAL TABLE PRACTICE.TABLE_TYPES.ext_sales (
--     sale_id    NUMBER    AS (VALUE:sale_id::NUMBER),
--     product    VARCHAR   AS (VALUE:product::VARCHAR),
--     amount     FLOAT     AS (VALUE:amount::FLOAT),
--     sale_date  DATE      AS (VALUE:sale_date::DATE)
-- )
-- PARTITION BY (sale_date)
-- WITH LOCATION = @PRACTICE.TABLE_TYPES.my_s3_stage
-- FILE_FORMAT  = (TYPE = 'PARQUET')
-- AUTO_REFRESH = TRUE;

SELECT 'External Table: Requires external stage (S3/GCS/Azure). See commented script above.' AS NOTE;

---
## 5. Hybrid Table

In [ ]:
%%sql -r hybrid_tbl
CREATE OR REPLACE HYBRID TABLE PRACTICE.TABLE_TYPES.ht_customers (
    customer_id   NUMBER        PRIMARY KEY,
    first_name    VARCHAR(100)  NOT NULL,
    last_name     VARCHAR(100)  NOT NULL,
    email         VARCHAR(255)  UNIQUE,
    phone         VARCHAR(20),
    created_at    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    INDEX idx_last_name (last_name),
    INDEX idx_phone (phone) INCLUDE (first_name, email)
)
COMMENT = 'OLTP customer table with enforced PK, unique email, and secondary indexes';

INSERT INTO PRACTICE.TABLE_TYPES.ht_customers (customer_id, first_name, last_name, email, phone)
VALUES
(1, 'Sai', 'Teja', 'sai.teja@example.com', '555-0101'),
(2, 'Neelu', 'Admin', 'neelu@example.com', '555-0102');

SELECT * FROM PRACTICE.TABLE_TYPES.ht_customers;

---
## 6. Iceberg Table

In [ ]:
%%sql -r iceberg_tbl
-- Iceberg table requires an external volume.
-- CREATE OR REPLACE EXTERNAL VOLUME my_iceberg_vol
--     STORAGE_LOCATIONS = (
--         (
--             NAME = 'my-s3-loc'
--             STORAGE_BASE_URL = 's3://my-bucket/iceberg/'
--             STORAGE_PROVIDER = 'S3'
--             STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::123456789012:role/my-role'
--         )
--     );

-- CREATE OR REPLACE ICEBERG TABLE PRACTICE.TABLE_TYPES.ice_products (
--     product_id    NUMBER,
--     product_name  VARCHAR(200),
--     category      VARCHAR(100),
--     price         DECIMAL(10,2),
--     stock_qty     NUMBER,
--     created_at    DATE DEFAULT CURRENT_DATE(),
--     is_active     BOOLEAN DEFAULT TRUE
-- )
-- CATALOG         = 'SNOWFLAKE'
-- EXTERNAL_VOLUME = 'my_iceberg_vol'
-- BASE_LOCATION   = 'products/'
-- COMMENT         = 'Iceberg product catalog — open format, full DML support';

SELECT 'Iceberg Table: Requires external volume. See commented script above.' AS NOTE;

In [ ]:
%%sql -r ice_insert
-- INSERT — Add rows (same syntax as regular tables)
-- INSERT INTO PRACTICE.TABLE_TYPES.ice_products (product_id, product_name, category, price, stock_qty)
-- VALUES
-- (1, 'Mountain Bike Pro', 'Bicycles', 1299.99, 45),
-- (2, 'Road Bike Elite', 'Bicycles', 1899.99, 30),
-- (3, 'Bike Helmet X1', 'Accessories', 89.99, 200),
-- (4, 'Cycling Shorts', 'Apparel', 59.99, 150),
-- (5, 'Chain Lubricant', 'Maintenance', 12.99, 500);

SELECT 'INSERT: Standard INSERT INTO syntax — same as permanent tables' AS OPERATION;

In [ ]:
%%sql -r ice_update
-- UPDATE — Modify existing rows
-- UPDATE PRACTICE.TABLE_TYPES.ice_products
-- SET price = price * 1.10, stock_qty = stock_qty - 5
-- WHERE category = 'Bicycles';

SELECT 'UPDATE: Standard UPDATE SET syntax — fully supported on Snowflake-managed Iceberg' AS OPERATION;

In [ ]:
%%sql -r ice_delete
-- DELETE — Remove rows
-- UPDATE PRACTICE.TABLE_TYPES.ice_products SET is_active = FALSE WHERE product_id = 5;
-- DELETE FROM PRACTICE.TABLE_TYPES.ice_products WHERE is_active = FALSE;

SELECT 'DELETE: Standard DELETE FROM syntax — fully supported' AS OPERATION;

In [ ]:
%%sql -r ice_merge
-- MERGE — Upsert (Insert + Update in one statement)
-- CREATE OR REPLACE TEMPORARY TABLE PRACTICE.TABLE_TYPES.stg_products AS
-- SELECT * FROM VALUES
--     (2, 'Road Bike Elite V2', 'Bicycles', 1999.99, 25, CURRENT_DATE(), TRUE),
--     (6, 'Water Bottle', 'Accessories', 14.99, 300, CURRENT_DATE(), TRUE)
-- AS t(product_id, product_name, category, price, stock_qty, created_at, is_active);
--
-- MERGE INTO PRACTICE.TABLE_TYPES.ice_products AS tgt
-- USING PRACTICE.TABLE_TYPES.stg_products AS src
-- ON tgt.product_id = src.product_id
-- WHEN MATCHED THEN UPDATE SET tgt.product_name = src.product_name, tgt.price = src.price
-- WHEN NOT MATCHED THEN INSERT VALUES (src.product_id, src.product_name, src.category, src.price, src.stock_qty, src.created_at, src.is_active);

SELECT 'MERGE: Full upsert support — same MERGE INTO syntax as permanent tables' AS OPERATION;

In [ ]:
%%sql -r ice_timetravel
-- TIME TRAVEL — Query past versions of data
-- SELECT * FROM PRACTICE.TABLE_TYPES.ice_products AT(OFFSET => -60*5);
-- SELECT * FROM PRACTICE.TABLE_TYPES.ice_products AT(TIMESTAMP => '2026-04-09 10:00:00'::TIMESTAMP_NTZ);
-- SELECT * FROM PRACTICE.TABLE_TYPES.ice_products BEFORE(STATEMENT => '<query_id>');
-- DROP TABLE PRACTICE.TABLE_TYPES.ice_products;
-- UNDROP TABLE PRACTICE.TABLE_TYPES.ice_products;

SELECT 'TIME TRAVEL: AT(OFFSET), AT(TIMESTAMP), BEFORE(STATEMENT), UNDROP all supported' AS OPERATION;

In [ ]:
%%sql -r ice_schema
-- SCHEMA EVOLUTION — Add/Drop/Rename columns (metadata-only, no data rewrite)
-- ALTER ICEBERG TABLE PRACTICE.TABLE_TYPES.ice_products ADD COLUMN weight_kg FLOAT;
-- ALTER ICEBERG TABLE PRACTICE.TABLE_TYPES.ice_products RENAME COLUMN stock_qty TO inventory_count;
-- ALTER ICEBERG TABLE PRACTICE.TABLE_TYPES.ice_products DROP COLUMN weight_kg;
-- Spark/Flink/Trino will automatically see the updated schema!

SELECT 'SCHEMA EVOLUTION: ADD, RENAME, DROP columns — metadata-only, no data rewrite' AS OPERATION;

---
## 7. Permanent Table (Default)

In [ ]:
%%sql -r permanent_tbl
CREATE OR REPLACE TABLE PRACTICE.TABLE_TYPES.perm_employees (
    emp_id        NUMBER AUTOINCREMENT,
    first_name    VARCHAR(100) NOT NULL,
    last_name     VARCHAR(100) NOT NULL,
    department    VARCHAR(50),
    salary        DECIMAL(12,2),
    hire_date     DATE,
    is_active     BOOLEAN DEFAULT TRUE
)
DATA_RETENTION_TIME_IN_DAYS = 30
CHANGE_TRACKING = TRUE
ENABLE_SCHEMA_EVOLUTION = TRUE
COMMENT = 'Permanent table — 30-day time travel, CDC enabled, schema evolution on'
CLUSTER BY (department);

INSERT INTO PRACTICE.TABLE_TYPES.perm_employees (first_name, last_name, department, salary, hire_date)
VALUES
('Sai', 'Teja', 'Engineering', 95000, '2024-01-15'),
('Neelu', 'Admin', 'Data', 105000, '2023-06-01'),
('Ravi', 'Kumar', 'Engineering', 88000, '2025-03-10');

SELECT * FROM PRACTICE.TABLE_TYPES.perm_employees;

---
## 8. Temporary Table

In [ ]:
%%sql -r temporary_tbl
CREATE OR REPLACE TEMPORARY TABLE PRACTICE.TABLE_TYPES.tmp_staging (
    batch_id      NUMBER,
    raw_data      VARIANT,
    loaded_at     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
DATA_RETENTION_TIME_IN_DAYS = 0
COMMENT = 'Temporary session-scoped staging — auto-dropped on session end';

INSERT INTO PRACTICE.TABLE_TYPES.tmp_staging (batch_id, raw_data)
SELECT 1, PARSE_JSON('{"key": "value1"}')
UNION ALL
SELECT 2, PARSE_JSON('{"key": "value2"}');

SELECT *, 'This table will auto-drop when session ends!' AS WARNING
FROM PRACTICE.TABLE_TYPES.tmp_staging;

---
## 9. Transient Table

In [ ]:
%%sql -r transient_tbl
CREATE OR REPLACE TRANSIENT TABLE PRACTICE.TABLE_TYPES.trn_etl_staging (
    source_system   VARCHAR(50),
    table_name      VARCHAR(100),
    record_count    NUMBER,
    load_status     VARCHAR(20) DEFAULT 'PENDING',
    loaded_at       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
DATA_RETENTION_TIME_IN_DAYS = 1
CHANGE_TRACKING = TRUE
COMMENT = 'Transient ETL staging — 1-day time travel, no fail-safe cost';

INSERT INTO PRACTICE.TABLE_TYPES.trn_etl_staging (source_system, table_name, record_count, load_status)
VALUES
('SQLSERVER', 'CUSTOMERS', 1445, 'SUCCESS'),
('SQLSERVER', 'ORDERS', 2199, 'SUCCESS'),
('SQLSERVER', 'PRODUCTS', 321, 'PENDING');

SELECT * FROM PRACTICE.TABLE_TYPES.trn_etl_staging;

---
## Quick Comparison Cheat Sheet

| Feature | Permanent | Transient | Temporary | Dynamic | Hybrid | External | Iceberg | Event |
|---------|-----------|-----------|-----------|---------|--------|----------|---------|-------|
| **Persists across sessions** | Yes | Yes | No | Yes | Yes | Yes | Yes | Yes |
| **Time Travel (max days)** | 90 | 1 | 1 | 1 | 1 | 0 | 1 | 1 |
| **Fail-safe** | 7 days | No | No | 7 days | 7 days | No | 7 days | 7 days |
| **DML Support** | Full | Full | Full | No (auto) | Full | Read-only | Full | Insert-only |
| **Primary Key Enforced** | No | No | No | No | **Yes** | No | No | No |
| **Best For** | Production | Staging/ETL | Session temp | Pipelines | OLTP | Data lake | Lakehouse | Logging |